# Model Evaluation & Deployment

This notebook is the final step of our ML workflow:
- Select best model from the benchmark comparison (Notebook 10)
- Package the trained model for SageMaker inference
- Deploy a real time SageMaker endpoint
- Clean up resources

### Setup Environment

In [2]:
import boto3 
import sagemaker
import os, tarfile, pickle, json
import pandas as pd
import numpy as np
from datetime import datetime
import xgboost as xgb

from sagemaker.xgboost.model import XGBoostModel
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()

print("Bucket:", bucket)
print("Role:", role)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Bucket: sagemaker-us-east-2-869435311794
Role: arn:aws:iam::869435311794:role/service-role/AmazonSageMaker-ExecutionRole-20260125T160894


### Download Model + Preprocessing Artifacts from S3

In [3]:
s3 = boto3.client("s3")
deploy_d = "/tmp/deploy"
os.makedirs(deploy_d, exist_ok=True)

# XGBoost achieved a higher Macro F1 than logreg benchmark, so we deploy XGBoost model
best_mod = "xgboost"

# Download model.pkl
model_key = f"models/benchmarks/{best_mod}/model.pkl"
local_model_pkl = f"{deploy_d}/model.pkl"

s3.download_file(bucket, model_key, local_model_pkl)

# Download preprocessing artifacts
s3.download_file(bucket, "models/benchmarks/shared/scaler.pkl", f"{deploy_d}/scaler.pkl")
s3.download_file(bucket, "models/benchmarks/shared/label_encoder.pkl", f"{deploy_d}/label_encoder.pkl")
s3.download_file(bucket, "models/benchmarks/shared/feature_cols.json", f"{deploy_d}/feature_cols.json")

print("Model and preprocessing artifacts downloaded")

# Convery the pkl XGBoost model to a native XGBoost model file
with open(local_model_pkl, "rb") as f:
    loaded_model = pickle.load(f)

xgb_json = f"{deploy_d}/xgb_model.json"

if hasattr(loaded_model, "save_model"):
    loaded_model.save_model(xgb_json)
else:
    booster = loaded_model.get_booster()
    booster.save_model(xgb_json)
    
print("Saved model to:", xgb_json)

Model and preprocessing artifacts downloaded
Saved model to: /tmp/deploy/xgb_model.json


### Create inference.py

In [4]:
# Make sure /tmp/deploy exists
os.makedirs("/tmp/deploy", exist_ok=True)

In [5]:
%%writefile /tmp/deploy/inference.py

# Import libraries again inside inference (incase container crashes)
import os
import pickle
import json
import numpy as np
import pandas as pd
import xgboost as xgb

def model_fn(model_dir):
    model_path = os.path.join(model_dir, "xgb_model.json")
    booster = xgb.Booster()
    booster.load_model(model_path)

    # load preprocessing artifacts
    with open(os.path.join(model_dir, "scaler.pkl"), "rb") as f:
        scaler = pickle.load(f)
        
    with open(os.path.join(model_dir, "label_encoder.pkl"), "rb") as f:
        label_encoder = pickle.load(f)
        
    with open(os.path.join(model_dir, "feature_cols.json"), "r") as f:
        feature_cols = json.load(f)
        
    return {
        "booster": booster, 
        "scaler": scaler, 
        "label_encoder": label_encoder, 
        "feature_cols": feature_cols,
    }

def input_fn(request_body, request_content_type):
    if request_content_type != "application/json":
        raise ValueError(f"{request_content_type} not supported")

    payload = json.loads(request_body)
    
    # allow 1 record or multiple
    if isinstance(payload, dict):
        payload = [payload]
        
    return pd.DataFrame(payload)

def predict_fn(input_data, artifacts):
    """ Run prediction and decode label"""
    feature_cols = artifacts["feature_cols"]
    rows = input_data.to_dict("records")
    
    X = np.array([[row.get(col, 0) for col in feature_cols] for row in rows], dtype=float)
    X_scaled = artifacts["scaler"].transform(X)

    dmatr = xgb.DMatrix(X_scaled)
    preds = artifacts["booster"].predict(dmatr)

    if getattr(preds, "ndim", 1) > 1:
        pred_labels = np.argmax(preds, axis=1) 
    else:
        pred_labels = (preds >= 0.5).astype(int)
        
    # Try to decode class labels, or return
    try:
        return artifacts["label_encoder"].inverse_transform(pred_labels).tolist()
    except Exception:
        return pred_labels.tolist()

def output_fn(prediction, response_content_type):
    if response_content_type != "application/json":
        raise ValueError(f"{response_content_type} not supported")
    return json.dumps({"Predictions": prediction})

Overwriting /tmp/deploy/inference.py


### Package Model into model.tar.gz

In [6]:
# Create model.tar.gz
path_t = "/tmp/model.tar.gz"
with tarfile.open(path_t, "w:gz") as tar:
    tar.add("/tmp/deploy/xgb_model.json", arcname="xgb_model.json")
    tar.add("/tmp/deploy/scaler.pkl", arcname="scaler.pkl")
    tar.add("/tmp/deploy/label_encoder.pkl", arcname="label_encoder.pkl")
    tar.add("/tmp/deploy/feature_cols.json", arcname="feature_cols.json")

print("Created:", path_t)

Created: /tmp/model.tar.gz


### Upload Model Artifact to S3

In [7]:
deploy_k = f"models/deploy/{best_mod}/model.tar.gz"
s3.upload_file(path_t, bucket, deploy_k)

data_mod = f"s3://{bucket}/{deploy_k}"

print("Model Data:", data_mod)

Model Data: s3://sagemaker-us-east-2-869435311794/models/deploy/xgboost/model.tar.gz


### Deploy Real-Time Endpoint

In [8]:
xgb_model = XGBoostModel(
    model_data=data_mod,
    role=role,
    entry_point="inference.py",
    source_dir="/tmp/deploy",
    py_version="py3",
    framework_version="1.7-1",
    sagemaker_session=sess
)

endpoint_name = f"aai540-group-7-{best_mod}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

predictor = xgb_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name
)

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

print("Deployed Endpoint:", endpoint_name)

------!Deployed Endpoint: aai540-group-7-xgboost-20260205-023735


In [12]:
# To test, load feature names first
with open("/tmp/deploy/feature_cols.json") as f:
    feature_cols = json.load(f)
feature_cols

['meanfreq',
 'sd',
 'median',
 'q25',
 'q75',
 'iqr',
 'skew',
 'kurt',
 'sp_ent',
 'sfm',
 'mode',
 'centroid',
 'meanfun',
 'minfun',
 'maxfun',
 'meandom',
 'mindom',
 'maxdom',
 'dfrange',
 'modindx']

In [13]:
# Create test 
test = {col: 0 for col in feature_cols}
test

# Call endpoint
prediction = predictor.predict(test)
prediction

{'Predictions': ['Sad']}

In [19]:
# Test batch (not real values)
test_batch = [
    {col: 0 for col in feature_cols},
    {col: 10 for col in feature_cols},
    {col: 50 for col in feature_cols}
]
predictor.predict(test_batch)

{'Predictions': ['Sad', 'Sad', 'Sad']}

### Delete Endpoint

In [20]:
predictor.delete_endpoint(delete_endpoint_config=True)
print("Deleted")

Deleted
